# Quantization: what int8 and int4 actually cost

**The trade curve is not a line, and it has a cliff** · GPU · ~40 min · Colab

Quantization is the cheapest win in deployment: store the weights in fewer bits, and the model fits on hardware that could not hold it before. The papers tell you it works. What they cannot tell you is what YOUR model loses, because the answer depends on the model, the data, and where in the network the precision was spent.

So the interesting question is not "does int4 work" — it is what the curve between fp16 and int4 looks like. Read most summaries and you would expect a gentle slope. Measure it and you find something closer to a plateau followed by an edge, and the edge is not where the bit count halves.

### The goal

Measure perplexity, memory and latency for the same model at fp16, int8 and int4, plot the three against each other, and identify which of the three costs moves first and which barely moves at all.

### The papers behind this

- [llm-int8](https://azimuth.plus/en/paper/llm-int8) — why naive int8 breaks large models, and the outlier insight that fixes it — Dettmers et al., 2022
- [gptq](https://azimuth.plus/en/paper/gptq) — one-shot 4-bit quantization that survives, by minimising layer-wise error — Frantar et al., 2023

> Save a copy to Drive before you start (File → Save a copy in Drive). Edits to the original are not saved.

## Setup

`PROFILE` is the only scale knob. The free tier is the default and stays inside Colab's free envelope.

In [ ]:
SLUG = "quantization-what-it-costs"
LANG = "en"
PROFILE = "free"  # free | a100

# Colab defaults to inline figures; CI does not. Being explicit means the
# captured plot on the site and the plot you see are produced the same way.
%matplotlib inline

# The shim lives in this repository, not on PyPI: a workshop should never
# depend on a package index staying up.
import os
import subprocess
import sys
from pathlib import Path

# Guarded three ways. Colab users re-run the setup cell constantly, and
# a contributor may already be sitting inside a checkout — the first
# Windows run of this notebook cloned the repository into its own
# generated/notebooks/ directory because neither case was handled.
REPO = 'azimuth-workshops'
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'shim' / 'azimuth_nb').is_dir()), None)
if root is None:
    if not Path(REPO).exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', 'stable',
             'https://github.com/MotazSabri/azimuth-workshops.git', REPO],
            check=True,
        )
    root = (here / REPO).resolve()
print('workshop root:', root)

# Absolute, so nothing depends on the working directory. Dropping the
# %cd this cell used to do also means the notebook stops caring where
# it was opened from.
sys.path.insert(0, str(root / 'shim'))
_ = os.environ.setdefault('AZIMUTH_DATA_DIR', str(root / 'data'))

In [ ]:
# Declared by this workshop (dependencies: in workshop.yaml).
# torch is NOT installed here — it is asserted, because a second
# torch over Colab's own will not match the driver.
%pip install -q transformers==4.44.2 bitsandbytes==0.43.3 accelerate==0.33.0 datasets==2.21.0

Three numbers move when you quantize, and they do not move together. Memory falls roughly with the bit count, and that part is arithmetic. Quality falls too, but not smoothly — it barely moves for a while and then falls off. Latency is the one that surprises people, because it can go the wrong way entirely.

Measuring all three at once is the point. A paper reporting only perplexity is not hiding anything; it is answering a different question from the one you have when you are deciding what to deploy.

_Preflight. This is the first workshop here that genuinely needs a GPU — int8 and int4 kernels are CUDA only._

In [ ]:
import azimuth_nb as azimuth

env = azimuth.setup(SLUG, lang=LANG, profile=PROFILE)

_Build the evaluation windows ONCE, before any model loads. Every variant is scored on identical text — a different sample per model would hide a real regression inside sampling noise._

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL = env.cfg["model"]
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# BUILD THE WINDOWS ONCE, BEFORE ANY MODEL LOADS.
#
# Every variant is scored on byte-identical text. Re-sampling per model would
# put sampling noise on the same axis as the effect being measured, and a real
# int4 regression could hide inside it — the numbers would still look precise.
raw = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
text = "\n\n".join(t for t in raw["text"] if t.strip())
all_ids = tokenizer(text, return_tensors="pt").input_ids[0][: env.cfg["evalTokens"]]

WINDOW = 512
windows = [all_ids[i : i + WINDOW] for i in range(0, len(all_ids) - WINDOW, WINDOW)]
n_windows = len(windows)

if env.lang == "ar":
    print(f"النموذج: {MODEL}")
    print(f"نوافذ التقييم: {n_windows} نافذة × {WINDOW} رمز")
else:
    print(f"model: {MODEL}")
    print(f"eval windows: {n_windows} × {WINDOW} tokens")

> **The paper** · [llm-int8](https://azimuth.plus/en/paper/llm-int8) — why naive int8 breaks large models, and the outlier insight that fixes it — Dettmers et al., 2022
>
> Dettmers and colleagues found that naive int8 collapses on large models not because 8 bits is too few on average, but because a handful of outlier feature dimensions have a range the rest do not. Keeping those in higher precision is the whole fix, and it is why int8 below is nearly free while a naive implementation would not be.

_Three loads, three measurements each. The model is freed between variants so the memory figure is the model, not the leftovers._

In [ ]:
import gc
import time

from transformers import AutoModelForCausalLM, BitsAndBytesConfig


def load(variant):
    """One model, three ways. Only the dtype/quantization config differs."""
    if variant == "fp16":
        kwargs = {"torch_dtype": torch.float16}
    elif variant == "int8":
        kwargs = {"quantization_config": BitsAndBytesConfig(load_in_8bit=True)}
    else:
        kwargs = {
            "quantization_config": BitsAndBytesConfig(
                load_in_4bit=True,
                # NF4 rather than plain int4: GPTQ's lesson is that WHERE the
                # levels sit matters as much as how many there are.
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
        }
    return AutoModelForCausalLM.from_pretrained(MODEL, device_map="cuda:0", **kwargs)


def perplexity(model):
    """Mean NLL over the shared windows, exponentiated."""
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for window in windows:
            ids = window.unsqueeze(0).to("cuda:0")
            loss = model(ids, labels=ids).loss
            total += loss.item() * ids.numel()
            count += ids.numel()
    return float(torch.exp(torch.tensor(total / count)))


def footprint(model):
    """Megabytes of parameters as actually stored, not as declared.

    Reading nelement * element_size per parameter is the honest measure: a
    4-bit tensor reports element_size 1 with two values packed per byte, so
    counting declared dtypes would overstate int4 by exactly the factor the
    workshop is trying to measure.
    """
    return sum(p.nelement() * p.element_size() for p in model.parameters()) / 1024**2


def latency(model, runs, new_tokens):
    """Median milliseconds per generated token, single sequence."""
    prompt = windows[0][:64].unsqueeze(0).to("cuda:0")
    with torch.no_grad():  # warm the kernels before timing anything
        model.generate(prompt, max_new_tokens=8, do_sample=False)
    torch.cuda.synchronize()
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        with torch.no_grad():
            model.generate(prompt, max_new_tokens=new_tokens, do_sample=False)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000 / new_tokens)
    times.sort()
    return times[len(times) // 2]


results = {}
for variant in ("fp16", "int8", "int4"):
    model = load(variant)
    results[variant] = {
        "ppl": perplexity(model),
        "mb": footprint(model),
        "ms": latency(model, env.cfg["latencyRuns"], env.cfg["newTokens"]),
    }
    print(
        f"  {variant:5} ppl {results[variant]['ppl']:7.3f}"
        f"  {results[variant]['mb']:8.1f} MB"
        f"  {results[variant]['ms']:6.1f} ms/token"
    )
    # Freed between variants so the memory figure is the model, not leftovers.
    del model
    gc.collect()
    torch.cuda.empty_cache()

fp16_ppl, int8_ppl, int4_ppl = (results[v]["ppl"] for v in ("fp16", "int8", "int4"))
fp16_mb, int8_mb, int4_mb = (results[v]["mb"] for v in ("fp16", "int8", "int4"))

> **On scale** — The free profile uses {{scale.model}} and {{scale.evalTokens}} tokens of evaluation text — small enough to load three times on a T4 without reconnecting. The shape of the curve is what transfers to a larger model, not the absolute perplexity.

_Three costs on one figure. Read which line is flat and which has a corner — the corner is the decision, and it is not where the bit count halves._

In [ ]:
import matplotlib.pyplot as plt

memory_ratio = fp16_mb / int4_mb
ppl_ratio_int8 = int8_ppl / fp16_ppl
ppl_ratio_int4 = int4_ppl / fp16_ppl

variants = ["fp16", "int8", "int4"]
fig, axes = plt.subplots(1, 3, figsize=(9, 2.8))
for ax, key, title in zip(
    axes,
    ["ppl", "mb", "ms"],
    ["perplexity", "memory (MB)", "ms / token"],
):
    values = [results[v][key] for v in variants]
    ax.plot(variants, values, marker="o", color="#457b9d")
    ax.set_title(title, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    # Zero-based so a flat line LOOKS flat. Autoscaled axes turn a 1% change
    # into a dramatic slope, which is how a chart lies without a wrong number
    # anywhere in it.
    ax.set_ylim(0, max(values) * 1.2)
fig.tight_layout()
plt.show()

if env.lang == "ar":
    print(f"الذاكرة: النصفية أكبر بـ {memory_ratio:.2f}× من الرباعية")
    print(f"الحيرة: ثمانية {ppl_ratio_int8:.3f}× · رباعية {ppl_ratio_int4:.3f}×")
else:
    print(f"memory: fp16 is {memory_ratio:.2f}× larger than int4")
    print(f"perplexity: int8 {ppl_ratio_int8:.3f}× · int4 {ppl_ratio_int4:.3f}×")

_The one that goes the wrong way. Smaller weights, more work per matmul._

In [ ]:
fp16_ms, int8_ms, int4_ms = (results[v]["ms"] for v in ("fp16", "int8", "int4"))

# Report the DIRECTION, because the direction is the surprise. Quantized
# weights are smaller but every matmul now pays a dequantize step, so at batch
# size 1 the smaller model is often the slower one.
if env.lang == "ar":
    print(
        f"زمن الاستجابة: نصفية {fp16_ms:.1f} · ثمانية {int8_ms:.1f} · رباعية {int4_ms:.1f} م.ث/رمز"
    )
    verdict = "أبطأ" if int4_ms > fp16_ms else "أسرع"
    print(
        f"الرباعية {verdict} من النصفية بعامل {max(int4_ms, fp16_ms) / min(int4_ms, fp16_ms):.2f}"
    )
else:
    print(f"latency: fp16 {fp16_ms:.1f} · int8 {int8_ms:.1f} · int4 {int4_ms:.1f} ms/token")
    verdict = "SLOWER" if int4_ms > fp16_ms else "faster"
    print(f"int4 is {verdict} than fp16 by {max(int4_ms, fp16_ms) / min(int4_ms, fp16_ms):.2f}×")

### Exercise — pick-a-deployment

Choose a variant for a stated deployment, and defend it with your own numbers rather than the defaults. Write down the constraint first — "fits in 8 GB", "under 50 ms per token", "quality within 2% of fp16" — then read the table and pick.

Then notice what the exercise did to you. Given one number you optimise it; given three you have to state a constraint, and the constraint is a product decision, not a technical one. Which of the three would you never trade, and would your answer change for a chatbot versus a batch summarizer?

_A hint is available: `env.hint(2)`_

In [ ]:
# YOUR TURN.
#
# State the constraint BEFORE reading the table. Then let the table choose.
BUDGET_MB = 400
MAX_MS_PER_TOKEN = 60
MAX_PPL_RATIO = 1.05

viable = [
    v
    for v in variants
    if results[v]["mb"] <= BUDGET_MB
    and results[v]["ms"] <= MAX_MS_PER_TOKEN
    and results[v]["ppl"] / fp16_ppl <= MAX_PPL_RATIO
]
print(f"viable under your constraints: {viable or 'none — loosen one, and say which'}")

_Memory must actually fall, and int8 must actually survive. If int8 moved a lot, hint 3 is the reason._

In [ ]:
memory_ok = env.check("memory-falls", memory_ratio)
quality_ok = env.check("quality-survives-int8", ppl_ratio_int8)

In [ ]:
receipt = env.receipt()